# 

In [20]:
from pathlib import Path
from datetime import datetime

import numpy as np

from dwd_radolan_utils.extraction import extract_time_series_from_radar
from dwd_radolan_utils.geo_utils import get_wgs84_grid
from dwd_radolan_utils.download import download_dwd


### 1. Configuration

In [21]:
DOWNLOAD_NEW_DATA = False
PRE_NAME = "col_name"

start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)

OUTPUT_NAME = f"ts_{start_date.strftime('%Y-%m-%d')}_{end_date.strftime('%Y-%m-%d')}"

radar_data_dir = Path("../data/dwd/processed_data")
radar_data_dir.mkdir(parents=True, exist_ok=True)

target_lat = 51.994682
target_lon = 6.915830

### 2. Download the historical RADOLAN data
**Note:** You only need to run this once. For 5 years of historical data, downloading might take a while.

In [22]:
if DOWNLOAD_NEW_DATA: 
    download_dwd(type_radolan="historical", start=start_date, end=end_date, save_path=radar_data_dir)

### 3. Find the RADOLAN grid pixel closest to your coordinate

In [23]:
wgs84_grid = get_wgs84_grid()  # Output shape: (900, 900, 2), where index 0 is latitude and 1 is longitude
distances = (wgs84_grid[:, :, 0] - target_lat) ** 2 + (wgs84_grid[:, :, 1] - target_lon) ** 2
closest_idx = np.unravel_index(np.argmin(distances), distances.shape)

print(f"Closest RADOLAN grid index to ({target_lat}, {target_lon}) is: {closest_idx}")

Closest RADOLAN grid index to (51.994682, 6.91583) is: (np.int64(565), np.int64(287))


### 4. Create the boolean grid mask (3D array expected by the extraction script)
Shape: (number_of_locations, 900, 900) - For our single location, it's (1, 900, 900)

In [24]:
grid = np.zeros((1, 900, 900), dtype=bool)

# Define a 3x3 grid around the closest coordinate
row_idx, col_idx = closest_idx
row_start, row_end = max(0, row_idx - 1), min(900, row_idx + 2)
col_start, col_end = max(0, col_idx - 1), min(900, col_idx + 2)

# Set the boolean mask to True at our identified 3x3 pixel area
grid[0, row_start:row_end, col_start:col_end] = True
print(f"Selected 3x3 area: rows {row_start}:{row_end}, columns {col_start}:{col_end} {np.sum(grid)} (should be 9 for a full 3x3 area)")

Selected 3x3 area: rows 564:567, columns 286:289 9 (should be 9 for a full 3x3 area)


### 5. Extract the time series

In [ ]:
ts_array, timestamps = extract_time_series_from_radar(
    grid=grid,
    path=radar_data_dir,
    start_date=start_date,
    end_date=end_date,
    grid_aggregation_method="mean",  # takes the mean over the 3x3 grid
    save=True,
    save_path=Path(OUTPUT_NAME),
    save_column_names=[PRE_NAME],
)

print(f"Extracted {len(timestamps)} time steps into {OUTPUT_NAME}")


(1, 900, 900)


  0%|          | 0/11 [00:00<?, ?it/s]